
# CS 431: Data-Intensive Distributed Computing (Fall 2026)
# Assignment 1: Warmup

In this assignment, you'll make sure that you've correctly set up your computing environment.
You'll then complete a classic "Word Count" task.

You can think of "Word Count" as the "Hello World!" of Hadoop, Spark, etc.
The task is simple: We want to count the total number of times each word occurs (in a potentially large collection of text).
Typically, we want to sort by the counts in descending order so we can examine the most frequently occurring words.

Note that there are two small (deliberately introduced) bugs in this notebook.
Make sure you fix them.

## 1. Data Acquisition

For this assignment we're going to be working with the [Amazon reviews 2023 dataset](https://amazon-reviews-2023.github.io/).

We're going to be working with the [reviews](https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Health_and_Personal_Care.jsonl.gz) for `Health_and_Personal_Care`.

Download it (either by hand or ask your agent to do it).

Set the path to the location of the downloaded file below (no need to uncompress the file).

In [ ]:
# TODO: Set the path to your copy of Health_and_Personal_Care.jsonl.gz - below it is hard-coded to my machine
DATA_PATH = "/Users/jimmylin/workspace/teaching2026/data/Health_and_Personal_Care.jsonl.gz"

Make sure the file is actually there!

In [ ]:
!ls $DATA_PATH

## 2. Environment Setup

Assuming you've set up your computing environment [based on the course instructions](https://lintool.github.io/cs431-2026f/software.html), the following code snippet should "just work" to initialize Spark.

If it doesn't, you'll need to debug your computing environment.

In [ ]:
import findspark, os

# TODO: Set path accordingly - below it is hard-coded to my machine
os.environ["SPARK_HOME"] = "/Users/jimmylin/workspace/teaching2026/spark-4.2.0-bin-hadoop3"
findspark.init()

Start a local Spark session:

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Assignment1")
    .master("local[*]")            # Use all local cores
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)

spark

If you've gotten to here, congrats! Everything seems to have been set up and initialized properly!

## 3. Word Count with RDDs

Let's read the `Health_and_Personal_Care.jsonl.gz` file into an RDD.

In [ ]:
import json

sc = spark.sparkContext
lines_rdd = sc.textFile(DATA_PATH)
reviews_rdd = lines_rdd.map(lambda line: json.loads(line))

# By the time we get to here, "reviews_rdd" should refer to an RDD with the reviews file loaded.
# Let's count the lines.

reviews_rdd.count()

You should find that there are 494,121 reviews, which aligns with what it says on the [Amazon reviews 2023 dataset](https://amazon-reviews-2023.github.io/) website.

If you're not getting this value, something is wrong. Debug.

Let's look at the first review:

In [ ]:
reviews_rdd.take(1)

You should see a JSON structure.

We're going to run word count on the `text` field.

Next, clean and tokenize text, and then find the 10 most common words.

Tokenize the text by lowercasing the entire text and splitting by a single space (i.e., `" "`).
Remove empty tokens.

<font color="red">**Here, you'll have to write some code!**</font>

**Hints:**

- You _must_ use `flatMap` and other RDD operations in this step. If you're not, you're doing something wrong...
- At the end, you'll need to `collect` the output.


In [ ]:
# acell_13_1kk98 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

# By the time we get to here "sorted_counts" should already have the collected output, sorted by frequency in descending order.
# So we just print out the top-10.

for word, count in sorted_counts[:10]:
    print(f"{word}: {count}")

The most frequent words are actually stopwords... so these results aren't very interesting.

Nevertheless, let's save the output.
Run the cell below.

In [ ]:
import csv

with open("rdd_top10_stopwords_retained.csv", "w", newline="", encoding="utf-8") as output_file:
    writer = csv.writer(output_file, lineterminator="\n")
    writer.writerow(["word", "count"])
    writer.writerows(sorted_counts[:10])

Let's try again, but now with stopwords removed.

Same tokenization, but remove stopwords using `StopWordsRemover` (already imported for you below).

<font color="red">**Write the code that does this.**</font>


In [ ]:
# acell_13_2ax4z (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

import json
from pyspark.ml.feature import StopWordsRemover

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

# By the time we get to here "sorted_counts" should already have the collected output, sorted by frequency in descending order.
# So we just print out the top-10.

for word, count in sorted_counts[:10]:
    print(f"{word}: {count}")

Let's save the output also:

In [ ]:
import csv

with open("rdd_top10_stopwords_removed.csv", "w", newline="", encoding="utf-8") as output_file:
    writer = csv.writer(output_file, lineterminator="\n")
    writer.writerow(["word", "count"])
    writer.writerows(sorted_counts[:10])

**Question to answer**:

Look at the top-10 `sorted_counts` above. Are all of them actual words? If not, explain what happened.

<font color="red">**Write your answer to the above question here!**</font>

// acell_13_3hq81 (keep this id for tracking purposes)

Answer: Replace with your answer.

## 4. Word Count with DataFrames

Now, we're going to do the same thing, but with DataFrames instead of RDDs.

What's the difference, you ask? We'll cover it in lecture soon enough!

Let's read the `Health_and_Personal_Care.jsonl.gz` file into a DataFrame.


In [ ]:
reviews_df = spark.read.json(DATA_PATH)
reviews_df.count()

You should find that there are 494,121 reviews, which is the same as using RDDs.

If you're not getting this value, something is wrong. Debug.

Let's look at the first review:

In [ ]:
reviews_df.take(1)

Compare the RDD and DataFrame versions:

```python
reviews_rdd.take(1)
reviews_df.take(1)
```

How are they different?
No need to write anything down, just think about it.

Next, clean and tokenize text, and then find the 10 most common (i.e., frequently occurring) words.
Do exactly the same processing as word count with RDDs above, except here you must use DataFrames.

<font color="red">**Here, you'll have to write some code!**</font>

**Hints:**

- You _must_ use `explode` and other Spark DataFrame operations in this exercise.
- This exercise shouldn't take more than (roughly) half a dozen lines. If you find yourself writing more code, you're doing something wrong...

In [ ]:
# acell_14_1le38 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

from pyspark.sql import functions as f

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

# By the time we get to here "sorted_counts_df" already has the counts sorted by frequency in descending order.
# So we just show the top-10.

sorted_counts_df.show(10, truncate=False)

Again, the most frequent words are actually stopwords... so these results aren't very interesting.

Nevertheless, let's just save the output.
Run the cell below.

In [ ]:
import csv

with open("df_top10_stopwords_retained.csv", "w", newline="", encoding="utf-8") as output_file:
    writer = csv.writer(output_file, lineterminator="\n")
    writer.writerow(["word", "count"])
    writer.writerows(sorted_counts_df.select("word", "count").limit(20).collect())

Let's try again, but now with stopwords removed, using DataFrames.

Same tokenization, but remove stopwords using `StopWordsRemover` (already imported for you below).

<font color="red">**Write the code that does this.**</font>

**Hints:**

- You _must_ use `explode` and other Spark DataFrame operations in this exercise.
- The code for this exercise shouldn't be much longer than the DataFrame operations above. If you find yourself writing more code, you're doing something wrong...

In [ ]:
# acell_14_2hg62 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

from pyspark.ml.feature import StopWordsRemover
from pyspark.sql import functions as f

######### BEGIN STUDENT CODE #########

######### END STUDENT CODE #########

# By the time we get to here "sorted_counts_df" already has the counts sorted by frequency in descending order.
# So we just show the top-10.

sorted_counts_df.show(10, truncate=False)

Let's save the output also:

In [ ]:
import csv

with open("df_top10_stopwords_removed.csv", "w", newline="", encoding="utf-8") as output_file:
    writer = csv.writer(output_file, lineterminator="\n")
    writer.writerow(["word", "count"])
    writer.writerows(sorted_counts_df.select("word", "count").limit(20).collect())

**Questions to reflect on**:

- What is conceptually different about how Spark executes `flatMap` and `explode`?
- What are the advantages or disadvantages of using each of them? 
- Are there cases where you may prefer one over the other?

(No need to write answers in the assignment submission. Just think about it...)

**Question to actually answer**:

Do the RDD approach and the DataFrame approach give the same answers? Explain why or why not.

<font color="red">**Write your answer to the above question here!**</font>

// acell_14_3ng52 (keep this id for tracking purposes)

Answer: Replace with your answer.

All done with Spark, so let's stop it.

In [ ]:
spark.stop()

## 5. Word Count with SQL

We're going to do the same thing, a third time, but using SQL.

Make sure that the server is running.


In [ ]:
# These are defaults, don't change them.
POSTGRES_HOST = "127.0.0.1"
POSTGRES_PORT = 5432

!pg_isready -h 127.0.0.1 -p 5432

If running the above command doesn't show that the PostgreSQL server is accepting connections, then something is wrong. Debug.

Also make sure that the user `cs431` exists.
If not, create the user.

In [ ]:
!psql -h 127.0.0.1 -d postgres -c "SELECT rolname, rolcanlogin, rolcreatedb FROM pg_roles WHERE rolname = 'cs431';"

If running the above cell doesn't show the `cs431` user, then it doesn't exist. Debug.

Next, write a script that loads the file at `DATA_PATH` into PostgreSQL:
+ Name the script `ingest_reviews.py`.
+ The script should take three arguments: the location of the source data, the name of the database, and the name of the table.
+ Hard-code the user to `cs431`.

<font color="red">**Write this script.**</font>

**Hint:** Ask your agent (to help).

Run the script below (but change the DB name):

In [ ]:
# TODO: Change the name of the database: change "lintool" to your GitHub username
POSTGRES_DB="cs431_a1_lintool"

! python ingest_reviews.py \
  $DATA_PATH \
  $POSTGRES_DB \
  reviews

Here's some boilerplate to connect to the database.

Just run the cell.

In [ ]:
import os
from urllib.parse import quote_plus

import pandas as pd
from sqlalchemy import create_engine, text

POSTGRES_USER = "cs431"
POSTGRES_PASSWORD = ""

password_fragment = f":{quote_plus(POSTGRES_PASSWORD)}" if POSTGRES_PASSWORD else ""
DATABASE_URL = (f"postgresql+psycopg://{quote_plus(POSTGRES_USER)}{password_fragment}@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}")

engine = create_engine(DATABASE_URL)

def read_sql(query, params=None):
    with engine.connect() as connection:
        return pd.read_sql(text(query), connection, params=params)

def execute_sql(statement, params=None):
    with engine.begin() as connection:
        connection.execute(text(statement), params or {})


As a sanity check, let's make sure everything is working:

In [ ]:
read_sql("SELECT current_database(), current_user, version() AS postgres_version")

Let's check the number of reviews below.

<font color="red">**Write the SQL query.**</font>

In [ ]:
# acell_15_1fw75 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

sql_query = """
-- BEGIN STUDENT CODE

-- END STUDENT CODE
"""

read_sql(sql_query)

Finally, you're going to perform word count on the `text` field in an SQL query.
Exactly the same as before (with RDDs and DataFrames), but using SQL.

Repeating:
Clean and tokenize text, and then find the 10 most common words.
Tokenize the text by lowercasing the entire text and splitting by a single space (i.e., `" "`).
Remove empty tokens.

<font color="red">**Write the SQL query.**</font>

In [ ]:
# acell_15_2umi4 (keep this id for tracking purposes)
# TODO: Write your code below, but do not remove any lines already in this cell.

sql_query = """
-- BEGIN STUDENT CODE

-- END STUDENT CODE
"""

# the two fields are (word, count)
results = read_sql(sql_query)
results

Let's save the output:

In [ ]:
results.to_csv("sql_top10_stopwords_retained.csv", index=False, encoding="utf-8", lineterminator="\n")

**Question to answer**:

Do the RDD approach and the SQL approach give the same answers? Explain why or why not.

<font color="red">**Write your answer to the above question here!**</font>

// acell_15_3er9s (keep this id for tracking purposes)

Answer: Replace with your answer.

## 6. Assignment Submission


**Final question to answer**:

What are the two small bugs in this notebook?

<font color="red">**Write your answer to the above question here!**</font>

// acell_16_1vb6x (keep this id for tracking purposes)

Answer: Replace with your answer.

For assignment submission details, see [the course homepage](https://lintool.github.io/cs431-2026f/assignments/assignment1.html).
Make sure you conform to the requirements articulated there.